In [ ]:
# pdf -> embedding model -> vector -> vector store

# query -> embedding -> vector -> similarity search -> top k -> model -> response

In [15]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [16]:
parser = StrOutputParser()

In [4]:
model = init_chat_model("openai:gpt-4o-mini")
emdeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

In [ ]:
doc = [
    Document(page_content="Artificial Intelligence (AI) is a branch of computer science focused on building systems capable of performing tasks that typically require human intelligence"),
    Document(page_content="Machine Learning (ML): A subset of AI where computers learn from experience without being explicitly programmed. The more data they process, the better they get at predicting outcomes or recognizing patterns."),
    Document(page_content="Deep Learning (DL): A more complex subset using layered neural networks inspired by the human brain, used for things like image and speech recognition."),
    Document(page_content="Natural Language Processing (NLP): Enables machines to read, understand, and generate human language."),
]

In [6]:
vector_store = InMemoryVectorStore(embedding=emdeddings)
vector_store.add_documents(doc)

['79cedf5a-1b98-4176-af2d-e6639a0f9e0d',
 '8497ced8-36b9-4e6d-ba27-b76d28db18c0',
 '720f3bad-f63d-4061-a91c-79583d395429',
 '99a19a85-e88b-4cc4-a8e3-d2daafeaa9c6']

In [8]:
question = "What is ML vs DL"

In [9]:
retrieved_docs = vector_store.similarity_search(question, k=2)

In [10]:
retrieved_docs

[Document(id='720f3bad-f63d-4061-a91c-79583d395429', metadata={}, page_content='Deep Learning: A more complex subset using layered neural networks inspired by the human brain, used for things like image and speech recognition.'),
 Document(id='8497ced8-36b9-4e6d-ba27-b76d28db18c0', metadata={}, page_content='Machine Learning: A subset of AI where computers learn from experience without being explicitly programmed. The more data they process, the better they get at predicting outcomes or recognizing patterns.')]

In [11]:
context = "\n".join(doc.page_content for doc in retrieved_docs)

In [12]:
context

'Deep Learning: A more complex subset using layered neural networks inspired by the human brain, used for things like image and speech recognition.\nMachine Learning: A subset of AI where computers learn from experience without being explicitly programmed. The more data they process, the better they get at predicting outcomes or recognizing patterns.'

In [13]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Only answer the question from the given context"),
    ("human", "Context: {context} \n\n Question: {question}")
])

In [17]:
chain = prompt | model | parser

In [18]:
response = chain.invoke({
    "context": context,
    "question": question
})

In [19]:
response

'Machine Learning (ML) is a subset of AI where computers learn from experience without being explicitly programmed, improving their ability to predict outcomes or recognize patterns through data processing. Deep Learning (DL) is a more complex subset of ML that uses layered neural networks, inspired by the human brain, and is often applied to tasks like image and speech recognition.'